# YOLOv3（PASCAL VOC 2007を用いた物体検出）

---
## 目的
YOLOv3は，オリジナルのYOLO（`yolo_v1.ipynb`）を大きく改良した1段階（single-shot）の物体検出手法です．オリジナルのYOLOが「7×7グリッド・1スケールの特徴マップ」だけで検出を行っていたのに対し，YOLOv3では次の2点が大きく変わりました．

1. **アンカーボックス（Anchor Box）の導入**：グリッドセルごとに直接ボックスの絶対座標を回帰するのではなく，あらかじめ用意した複数サイズの「アンカー」からの相対的なずれを回帰する（Faster R-CNNやSSDと同様の考え方）．
2. **複数スケールでの検出（マルチスケール予測）**：ネットワーク内の異なる解像度の特徴マップ3か所（浅い層〜深い層）それぞれで検出を行うことで，1スケールしか使わないYOLOv1の弱点であった小さい物体の検出精度を改善する．

さらに，クラス分類にはSoftmaxではなく，クラスごとに独立したSigmoid（多クラスの重複を許すマルチラベル的な扱い）を用いる点も特徴です．本ノートブックでは，PASCAL VOC 2007データセットを用いてYOLOv3をスクラッチに近い形で実装し，これらの仕組みを理解します．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, box_convert, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．`ssd.ipynb`と同じ`VOCDetectionDataset`をそのまま使用します．

`NUM_CLASSES`は`ssd.ipynb`の`n_class`（背景を含めた21クラス，Softmaxで分類）とは異なり，**背景クラスを持たない20クラス**とします．YOLOv3では，物体の有無は各アンカーの「objectness（物体らしさ）」スコアで独立に表現し，クラスは物体があるアンカーについてのみクラスごとに独立したSigmoidで予測するため，「背景」を1つのクラスとして数える必要がないためです（詳細は後述の損失関数の節を参照）．

入力画像サイズ`IMG_SIZE`は，3つの検出スケール（stride 8, 16, 32）で割り切れる必要があるため，原論文でも標準的に使われる416×416とします．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる（GTのラベル表現はssd.ipynbと共通化する）
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES)  # 20．背景クラスは持たない（objectnessスコアで物体の有無を表現する）
IMG_SIZE = 416  # 32の倍数（3つの検出スケールのstride 8/16/32で割り切れる必要がある）

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    '''VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する'''
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    '''VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset'''
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景を除く）:', NUM_CLASSES)

## バックボーン
原論文のYOLOv3は，Darknet-53という独自のバックボーンを使用しています．本ノートブックでは，他の検出ノートブックと同様に，事前学習済みのResNet50をバックボーンとして使用します．ResNet50から，解像度の異なる3つの特徴マップを取り出します．

- `layer2`の出力 (C3)：stride 8，52×52，channels 512
- `layer3`の出力 (C4)：stride 16，26×26，channels 1024
- `layer4`の出力 (C5)：stride 32，13×13，channels 2048

これら3つの特徴マップを，次節で説明するneck（特徴統合部）に入力し，3つのスケールでの検出を行います．

In [ ]:
class ConvBNAct(nn.Module):
    '''Darknet系でよく使われる Conv -> BatchNorm -> LeakyReLU(0.1) のブロック'''
    def __init__(self, in_ch, out_ch, k, s=1, p=None):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class YOLOv3Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x = self.stem(x)
        c3 = self.layer2(x)   # stride  8, 52x52, channels  512
        c4 = self.layer3(c3)  # stride 16, 26x26, channels 1024
        c5 = self.layer4(c4)  # stride 32, 13x13, channels 2048
        return c3, c4, c5

## Neck・検出ヘッド（マルチスケール特徴統合）
YOLOv3は，最も深い（解像度が低い）特徴マップから出発し，浅い特徴マップへ向かって情報を伝播させていくトップダウン構造のneckを持ちます．考え方はFPN（Feature Pyramid Network）に近いですが，YOLOv3独自の実装では，アップサンプリングした特徴マップと浅い層の特徴マップを**要素ごとの加算ではなくチャンネル方向に連結（concatenate）**する点が異なります．

具体的な流れは次の通りです．

1. C5に5層の畳み込み（1×1と3×3を交互に繰り返す「conv set」）を適用し，stride32スケール用の特徴（route1）を得る．route1からさらに3×3畳み込み＋1×1畳み込みで，1つ目の検出ヘッドの生の出力（raw prediction）を得る．
2. route1を1×1畳み込みでチャンネル削減した後，2倍にアップサンプリング（nearest neighbor）し，C4と**チャンネル方向に連結**する．これに再び5層のconv setを適用し，stride16スケール用の特徴（route2）を得て，2つ目の検出ヘッドの出力を得る．
3. route2についても同様に，アップサンプリング後C3と連結し，stride8スケール用の特徴（route3）から3つ目の検出ヘッドの出力を得る．

最終的に，stride8（52×52，高解像度）・stride16（26×26）・stride32（13×13，低解像度）の3つの生の予測テンソルが得られます．strideが小さい（解像度が高い）スケールほど小さい物体の検出に適し，strideが大きい（解像度が低い）スケールほど大きい物体の検出に適します（FPNの記法に合わせて呼ぶ場合，P3が最も解像度が高くstrideが小さいスケールに対応するため，「P3/P4/P5」と「stride 8/16/32」の対応関係は，数字の大小が逆になる点に注意してください）．

各検出ヘッドの出力チャンネル数は，1グリッドセル・1アンカーあたり「ボックスの4パラメータ + objectness 1 + クラススコア`NUM_CLASSES`個」で，アンカー数（1スケールあたり3個）をかけた`3 * (5 + NUM_CLASSES)`となります．

In [ ]:
NUM_ANCHORS_PER_SCALE = 3
PRED_CH = NUM_ANCHORS_PER_SCALE * (5 + NUM_CLASSES)  # (tx, ty, tw, th, t_obj, t_class_1...t_class_C) x 3anchors


def conv_set(in_ch, out_ch):
    '''1x1/3x3を交互に5層重ねる，YOLOv3のneckで繰り返し使われる畳み込みブロック'''
    return nn.Sequential(
        ConvBNAct(in_ch, out_ch, 1),
        ConvBNAct(out_ch, out_ch * 2, 3),
        ConvBNAct(out_ch * 2, out_ch, 1),
        ConvBNAct(out_ch, out_ch * 2, 3),
        ConvBNAct(out_ch * 2, out_ch, 1),
    )


class YOLOv3NeckHead(nn.Module):
    def __init__(self):
        super().__init__()
        # stride32スケール（大きい物体）
        self.set1 = conv_set(2048, 512)
        self.head1_conv = ConvBNAct(512, 1024, 3)
        self.head1_pred = nn.Conv2d(1024, PRED_CH, 1)
        self.up1_conv = ConvBNAct(512, 256, 1)

        # stride16スケール（中程度の物体）：C4とチャンネル連結
        self.set2 = conv_set(256 + 1024, 256)
        self.head2_conv = ConvBNAct(256, 512, 3)
        self.head2_pred = nn.Conv2d(512, PRED_CH, 1)
        self.up2_conv = ConvBNAct(256, 128, 1)

        # stride8スケール（小さい物体）：C3とチャンネル連結
        self.set3 = conv_set(128 + 512, 128)
        self.head3_conv = ConvBNAct(128, 256, 3)
        self.head3_pred = nn.Conv2d(256, PRED_CH, 1)

    def forward(self, c3, c4, c5):
        route1 = self.set1(c5)
        out1 = self.head1_pred(self.head1_conv(route1))  # (B, PRED_CH, 13, 13)

        up1 = F.interpolate(self.up1_conv(route1), scale_factor=2, mode='nearest')
        route2 = self.set2(torch.cat([up1, c4], dim=1))
        out2 = self.head2_pred(self.head2_conv(route2))  # (B, PRED_CH, 26, 26)

        up2 = F.interpolate(self.up2_conv(route2), scale_factor=2, mode='nearest')
        route3 = self.set3(torch.cat([up2, c3], dim=1))
        out3 = self.head3_pred(self.head3_conv(route3))  # (B, PRED_CH, 52, 52)

        return [out1, out2, out3]  # stride32, stride16, stride8 の順

## アンカー（Anchor）の設計
YOLOv3では，各スケールに3種類ずつ，合計9種類の「アンカー」（幅・高さの基準となるボックス）を用意します．原論文では，学習データのGTボックスの(幅, 高さ)分布に対してk-meansクラスタリングを行い，9個のアンカーサイズをデータドリブンに決定しています．本ノートブックでは実装を単純化し，一般に知られるYOLOv3のCOCO用アンカーサイズ（`IMG_SIZE=416`基準）を固定プリセットとしてそのまま用います．

小さいアンカー（`stride8`，高解像度スケール）は小さい物体，大きいアンカー（`stride32`，低解像度スケール）は大きい物体を検出する役割を担います．

In [ ]:
# 各スケールの3アンカー，(幅, 高さ) をピクセル単位で指定（IMG_SIZE=416基準の固定プリセット）
ANCHORS = [
    [(116, 90), (156, 198), (373, 326)],  # stride32 - 大きい物体用
    [(30, 61), (62, 45), (59, 119)],      # stride16 - 中程度の物体用
    [(10, 13), (16, 30), (33, 23)],       # stride8  - 小さい物体用
]
STRIDES = [32, 16, 8]

ANCHORS_T = [torch.tensor(a, dtype=torch.float32, device=device) for a in ANCHORS]
ALL_ANCHORS = torch.cat(ANCHORS_T, dim=0)  # (9, 2)：全スケールのアンカーをまとめたもの（マッチングで使用）

total_anchors = sum(NUM_ANCHORS_PER_SCALE * (IMG_SIZE // s) ** 2 for s in STRIDES)
print('アンカーの総数:', total_anchors)

## ボックスのエンコード・デコード
YOLOv3は，SSD（`ssd.ipynb`）のような分散（variance）を用いた正規化オフセットとは異なる，YOLOv3独自のSigmoidベースのパラメータ化を用います．グリッドセル$(c_x, c_y)$・アンカー（幅$p_w$，高さ$p_h$）ごとに，ネットワークは生の値$(t_x, t_y, t_w, t_h, t_{obj}, t_{class,1}, \dots, t_{class,C})$を出力し，次の式で画像上のボックスにデコードします．

- $b_x = \text{sigmoid}(t_x) + c_x$，$b_y = \text{sigmoid}(t_y) + c_y$（中心座標．Sigmoidでオフセットをセル内の範囲$[0, 1)$に制約してからグリッド座標を加える．中心が担当セルの外にはみ出さないようにする工夫で，SSDの分散スケール方式とはこの点で異なる）
- $b_w = p_w \cdot \exp(t_w)$，$b_h = p_h \cdot \exp(t_h)$（アンカーのサイズを基準とした指数スケール．考え方自体はSSDのデコードと同様だが，正規化座標のデフォルトボックスではなく，ピクセル単位のアンカーが基準になる）
- $\text{objectness} = \text{sigmoid}(t_{obj})$：物体らしさのスコア
- $\text{class score}_i = \text{sigmoid}(t_{class,i})$：クラス$i$のスコア（**クラスごとに独立したSigmoid**．Softmaxのようにクラス間で正規化しないため，複数クラスに同時に高いスコアを与えることもできる＝マルチラベル的な予測が可能）

以下では，推論時にネットワーク出力をデコードする`decode_scale`と，学習時にGTボックスをエンコードするための`encode_target`（次節のマッチングから呼び出す）を定義します．

In [ ]:
def logit(x, eps=1e-4):
    '''sigmoidの逆関数．学習ターゲットのt_x, t_yを計算するために使用する'''
    x = x.clamp(eps, 1 - eps)
    return torch.log(x / (1 - x))


def encode_target(gx, gy, gw, gh, anchor_wh, cell_x, cell_y):
    '''GTボックス（グリッド座標gx, gy，ピクセルサイズgw, gh）を，
    担当するアンカー・セルからの生のターゲット値 (tx, ty, tw, th) にエンコードする'''
    tx = logit(gx - cell_x)
    ty = logit(gy - cell_y)
    tw = torch.log(gw / anchor_wh[0] + 1e-9)
    th = torch.log(gh / anchor_wh[1] + 1e-9)
    return torch.stack([tx, ty, tw, th])


def decode_scale(raw, anchors, stride):
    '''1スケール分の生の予測テンソル (B, 3*(5+C), H, W) を，画像上のボックス座標・
    objectness・クラススコアにデコードする'''
    B, _, H, W = raw.shape
    num_anchors = anchors.shape[0]
    pred = raw.view(B, num_anchors, 5 + NUM_CLASSES, H, W).permute(0, 1, 3, 4, 2).contiguous()

    ys, xs = torch.meshgrid(torch.arange(H, device=raw.device), torch.arange(W, device=raw.device), indexing='ij')
    grid_x = xs.float().view(1, 1, H, W)
    grid_y = ys.float().view(1, 1, H, W)

    bx = (torch.sigmoid(pred[..., 0]) + grid_x) * stride
    by = (torch.sigmoid(pred[..., 1]) + grid_y) * stride
    aw = anchors[:, 0].view(1, num_anchors, 1, 1)
    ah = anchors[:, 1].view(1, num_anchors, 1, 1)
    bw = aw * torch.exp(pred[..., 2].clamp(max=10))  # expのオーバーフロー防止のためクランプ
    bh = ah * torch.exp(pred[..., 3].clamp(max=10))

    boxes_cxcywh = torch.stack([bx, by, bw, bh], dim=-1)
    boxes_xyxy = box_convert(boxes_cxcywh.view(-1, 4), 'cxcywh', 'xyxy').view(B, num_anchors, H, W, 4)
    boxes_xyxy = boxes_xyxy.clamp(min=0, max=IMG_SIZE)  # 画像範囲外に出たボックスをクリップする

    obj = torch.sigmoid(pred[..., 4])
    cls = torch.sigmoid(pred[..., 5:])  # クラスごとに独立したSigmoid（Softmaxではない）
    return boxes_xyxy, obj, cls

## アンカーとGTボックスのマッチング（Target Assignment）
SSD（`ssd.ipynb`）では，IoUが閾値以上となる**複数の**デフォルトボックスをまとめて正例として扱いました．YOLOv3のマッチングはこれよりもずっと疎（sparse）です．

1. 各GTボックスについて，(幅, 高さ)の形状だけを見たIoU（両方を原点中心に置いたと仮定して計算するIoU．位置は考慮しない）を，全9種類のアンカー（3スケール×3アンカー）との間で計算し，最もIoUが高いアンカーを1つ選ぶ．
2. そのアンカーが属するスケール上で，GTボックスの中心が入るグリッドセルだけが，そのGTに対する**唯一の正例**となる（ネットワーク全体で1つのGTにつき正例アンカーはちょうど1つ）．
3. それ以外のアンカーは，どのGTに対しても（そのアンカーの位置に置いた固定サイズのアンカーボックスとの）IoUが閾値（0.5）を下回っていれば負例，閾値を超えていれば「ignore（無視）」として扱う．ignoreに分類したアンカーは，正解でこそないもののGTにかなり近い予測をしている可能性が高いため，objectnessの損失計算から除外し，不当に「物体なし」を学習させないようにする．

SSDが「1つのGTに複数の正例」を許すのに対し，YOLOv3は「1つのGTに正例1つ＋ignoreによる緩衝地帯」という，より疎で厳格なマッチングを行う点が大きな違いです．

In [ ]:
def shape_iou(wh_a, wh_b):
    '''(幅, 高さ)だけを見た，両者を原点中心に置いたと仮定するIoU．wh_a: (N,2), wh_b: (M,2) -> (N,M)'''
    w_a, h_a = wh_a[:, 0:1], wh_a[:, 1:2]
    w_b, h_b = wh_b[:, 0], wh_b[:, 1]
    inter = torch.min(w_a, w_b) * torch.min(h_a, h_b)
    union = w_a * h_a + w_b * h_b - inter
    return inter / union.clamp(min=1e-9)


def build_targets(gt_boxes, gt_labels, ignore_threshold=0.5):
    '''1枚の画像内のGTボックスから，3スケール分の学習ターゲット（objectness, box, class, pos/ignoreマスク）を作成する'''
    grid_sizes = [IMG_SIZE // s for s in STRIDES]
    targets = [{
        'obj': torch.zeros(NUM_ANCHORS_PER_SCALE, g, g, device=device),
        'pos_mask': torch.zeros(NUM_ANCHORS_PER_SCALE, g, g, dtype=torch.bool, device=device),
        'ignore_mask': torch.zeros(NUM_ANCHORS_PER_SCALE, g, g, dtype=torch.bool, device=device),
        'box': torch.zeros(NUM_ANCHORS_PER_SCALE, g, g, 4, device=device),
        'cls': torch.zeros(NUM_ANCHORS_PER_SCALE, g, g, NUM_CLASSES, device=device),
    } for g in grid_sizes]

    if gt_boxes.shape[0] == 0:
        return targets

    gt_cxcywh = box_convert(gt_boxes, 'xyxy', 'cxcywh')

    # (1) 形状IoUが最大となる1つのアンカーを，各GTに割り当てる
    ious = shape_iou(gt_cxcywh[:, 2:], ALL_ANCHORS)  # (num_gt, 9)
    best_anchor = ious.argmax(dim=1)

    for i in range(gt_boxes.shape[0]):
        anchor_flat_idx = best_anchor[i].item()
        scale_idx = anchor_flat_idx // NUM_ANCHORS_PER_SCALE
        anchor_idx = anchor_flat_idx % NUM_ANCHORS_PER_SCALE
        stride = STRIDES[scale_idx]
        gsize = grid_sizes[scale_idx]

        cx, cy, w, h = gt_cxcywh[i]
        gx, gy = cx / stride, cy / stride
        cell_x = min(max(int(gx.item()), 0), gsize - 1)
        cell_y = min(max(int(gy.item()), 0), gsize - 1)

        tgt = targets[scale_idx]
        tgt['obj'][anchor_idx, cell_y, cell_x] = 1.0
        tgt['pos_mask'][anchor_idx, cell_y, cell_x] = True
        tgt['box'][anchor_idx, cell_y, cell_x] = encode_target(gx, gy, w, h, ANCHORS[scale_idx][anchor_idx], cell_x, cell_y)
        tgt['cls'][anchor_idx, cell_y, cell_x, gt_labels[i].item() - 1] = 1.0  # ラベルは1〜20 -> 0〜19に変換

    # (2) 正例ではないが，GTとのIoUが閾値を超えるアンカーはignoreとする（負例の学習対象から除外）
    for scale_idx, (stride, gsize, anchors) in enumerate(zip(STRIDES, grid_sizes, ANCHORS_T)):
        ys, xs = torch.meshgrid(torch.arange(gsize, device=device), torch.arange(gsize, device=device), indexing='ij')
        cx = ((xs.float() + 0.5) * stride).unsqueeze(0).expand(NUM_ANCHORS_PER_SCALE, -1, -1)
        cy = ((ys.float() + 0.5) * stride).unsqueeze(0).expand(NUM_ANCHORS_PER_SCALE, -1, -1)
        aw = anchors[:, 0].view(NUM_ANCHORS_PER_SCALE, 1, 1).expand(-1, gsize, gsize)
        ah = anchors[:, 1].view(NUM_ANCHORS_PER_SCALE, 1, 1).expand(-1, gsize, gsize)
        anchor_boxes = torch.stack([cx - aw / 2, cy - ah / 2, cx + aw / 2, cy + ah / 2], dim=-1).view(-1, 4)

        max_iou, _ = box_iou(anchor_boxes, gt_boxes).max(dim=1)
        max_iou = max_iou.view(NUM_ANCHORS_PER_SCALE, gsize, gsize)
        targets[scale_idx]['ignore_mask'] = (max_iou > ignore_threshold) & (~targets[scale_idx]['pos_mask'])

    return targets

## 損失関数
YOLOv3の損失は，次の4つの要素の和から構成されます．

- **Box regression loss**：正例のアンカーのみで計算する，生の予測値$(t_x, t_y, t_w, t_h)$と，対応するターゲット（`encode_target`で計算した値）とのSmooth L1損失．原論文は二乗誤差（SSE）を用いていますが，本ノートブックでは外れ値に強いSmooth L1を採用します（デコード後のピクセル座標ではなく，ネットワーク出力と同じ生の値の空間で計算する点に注意）．
- **Objectness loss**：正例（ターゲット1）と負例（ターゲット0）のアンカーで計算するBCE損失．ignoreに分類されたアンカーは計算対象から除外します．
- **Class loss**：正例のアンカーのみで計算する，クラスごとに独立したBCE損失（Softmaxではない）．

アンカーの大部分は負例（物体なし）であるため，SSDはHard Negative Miningで負例をサブサンプリングして正例:負例の比率を調整していました．YOLOv3では代わりに，objectnessの負例項に小さい重み`lambda_noobj`（0.5）を掛けることで，同じ「不均衡問題」に対処します．また，box regression lossには大きめの重み`lambda_coord`（5.0）を掛け，位置の精度を優先的に学習させます．

In [ ]:
class YOLOv3Loss(nn.Module):
    def __init__(self, n_classes, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.n_classes = n_classes
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.bce = nn.BCEWithLogitsLoss(reduction='sum')
        self.smooth_l1 = nn.SmoothL1Loss(reduction='sum')

    def forward(self, raw_preds, targets_batch):
        box_loss = raw_preds[0].new_tensor(0.0)
        obj_loss = raw_preds[0].new_tensor(0.0)
        noobj_loss = raw_preds[0].new_tensor(0.0)
        cls_loss = raw_preds[0].new_tensor(0.0)
        num_pos_total = raw_preds[0].new_tensor(0.0)

        for raw, tgt in zip(raw_preds, targets_batch):
            B, _, H, W = raw.shape
            pred = raw.view(B, NUM_ANCHORS_PER_SCALE, 5 + self.n_classes, H, W).permute(0, 1, 3, 4, 2).contiguous()
            pred_box, pred_obj, pred_cls = pred[..., :4], pred[..., 4], pred[..., 5:]

            pos = tgt['pos_mask']
            neg = (~pos) & (~tgt['ignore_mask'])  # ignoreは正例・負例どちらの損失計算からも除外する

            if pos.any():
                box_loss = box_loss + self.smooth_l1(pred_box[pos], tgt['box'][pos])
                obj_loss = obj_loss + self.bce(pred_obj[pos], tgt['obj'][pos])
                cls_loss = cls_loss + self.bce(pred_cls[pos], tgt['cls'][pos])
            if neg.any():
                noobj_loss = noobj_loss + self.bce(pred_obj[neg], torch.zeros_like(pred_obj[neg]))

            num_pos_total = num_pos_total + pos.sum()

        num_pos_total = num_pos_total.clamp(min=1)
        return (self.lambda_coord * box_loss + obj_loss + self.lambda_noobj * noobj_loss + cls_loss) / num_pos_total

## ネットワークの作成
これまでに定義した`YOLOv3Backbone`と`YOLOv3NeckHead`を組み合わせて，`YOLOv3`モデルを構築します．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用います．事前学習済みのバックボーンを使用するため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率を設定しています（`ssd.ipynb`と同じ設定です）。

In [ ]:
class YOLOv3(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = YOLOv3Backbone()
        self.neck_head = YOLOv3NeckHead()

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        return self.neck_head(c3, c4, c5)  # [stride32出力, stride16出力, stride8出力]


model = YOLOv3()
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = YOLOv3Loss(n_classes=NUM_CLASSES)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを8，学習エポック数を20とします．`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながら`build_targets`で3スケール分の学習ターゲットを作成し，バッチ方向にまとめてからネットワークに入力します。

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)

        # 画像ごとにターゲットを作成し，スケールごと・キーごとにバッチ方向へスタックする
        batch_targets = [{'obj': [], 'pos_mask': [], 'ignore_mask': [], 'box': [], 'cls': []} for _ in STRIDES]
        for t in targets:
            per_image_targets = build_targets(t['boxes'].to(device), t['labels'].to(device))
            for s, scale_tgt in enumerate(per_image_targets):
                for k, v in scale_tgt.items():
                    batch_targets[s][k].append(v)
        for s in range(len(STRIDES)):
            for k in batch_targets[s]:
                batch_targets[s][k] = torch.stack(batch_targets[s][k])

        raw_preds = model(images)

        loss = criterion(raw_preds, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（デコード・NMS）
学習したモデルで推論を行うための関数を定義します．3つのスケールそれぞれについて`decode_scale`でピクセル座標のボックスにデコードし，objectnessとクラススコアの積が閾値を超えるものだけを残します．3スケール分を1つに連結したのち，`torchvision.ops.nms`でクラスごとに重複した検出結果を除去（Non-Maximum Suppression）します。

In [ ]:
def predict(model, image_tensor, conf_threshold=0.3, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        raw_preds = model(image_tensor.unsqueeze(0).to(device))

    boxes_all, scores_all = [], []
    for raw, anchors, stride in zip(raw_preds, ANCHORS_T, STRIDES):
        boxes, obj, cls = decode_scale(raw, anchors, stride)
        scores = obj.unsqueeze(-1) * cls  # objectnessとクラススコアの積を信頼度とする
        boxes_all.append(boxes.reshape(-1, 4))
        scores_all.append(scores.reshape(-1, NUM_CLASSES))
    boxes_cat = torch.cat(boxes_all, dim=0)
    scores_cat = torch.cat(scores_all, dim=0)

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(NUM_CLASSES):
        class_scores = scores_cat[:, class_idx]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_cat[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        # GTのラベル表現（1〜20，parse_voc_target参照）に合わせるため class_idx (0〜19) + 1 とする
        result_labels.append(torch.full((len(keep),), class_idx + 1, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．算出には`ssd.ipynb`と同様に`torchmetrics`の`MeanAveragePrecision`を利用し，VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します。

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します。

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルYOLOv3との違い
本ノートブックで実装したYOLOv3は，原論文の構成を一部簡略化しています．主な違いは次の通りです．

| | 原論文（YOLOv3） | 本ノートブック |
|---|---|---|
| バックボーン | Darknet-53（スクラッチ学習） | ResNet50（ImageNet事前学習済み） |
| 入力画像サイズ | 320〜608の間で学習中にランダムに変更（マルチスケール学習） | 416×416固定 |
| アンカーサイズの決定方法 | 学習データのGTボックスに対するk-meansクラスタリング | 固定プリセット（一般に知られるCOCO用アンカーサイズを流用） |
| Ignoreアンカーの判定 | 各アンカー位置での現在の予測ボックスとGTのIoU（動的） | 各アンカー位置に置いた固定サイズのアンカーボックスとGTのIoU（静的，簡略化） |
| クラス数の設計 | 背景クラスなし，クラスごとに独立したSigmoid（本ノートブックと同じ） | 背景クラスなし，クラスごとに独立したSigmoid（変更なし） |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`・`box_iou`を使用 |
| mAP評価 | 独自実装（VOC/COCO形式のAP計算） | `torchmetrics`の`MeanAveragePrecision`を使用 |

特に，アンカーのマッチング（1GTにつき正例アンカー1つ＋ignore緩衝地帯）やSigmoidベースのボックスエンコーディングといったYOLOv3の核心部分は原論文通りに実装しており，バックボーンの置き換えとアンカーサイズの決定方法（k-means→固定プリセット）が主な簡略化点です。

## 課題

1. `ANCHORS`のプリセットを変更（各スケールのサイズ比を変える，あるいはスケール間のサイズが逆転するように変えるなど）し，学習の様子や検出結果にどのような影響が出るか確認しましょう．

2. `build_targets`の`ignore_threshold`（0.5）を変更し，ignoreとして除外されるアンカー数（`ignore_mask`の合計）や，学習の安定性（`noobj`項の損失の挙動）への影響を確認しましょう．

3. `build_targets`で1枚の画像あたり実際に正例となるアンカー数（`pos_mask`の合計）を数え，`ssd.ipynb`の`match_default_boxes`で正例となるデフォルトボックス数と比較しましょう．YOLOv3の「1GTにつき正例1つ」というマッチング戦略が，SSDの「IoU閾値以上を全て正例にする」戦略よりもどれだけ疎であるか，具体的な数字で確認しましょう．